# Ejecución del pipeline de preparación - MMOTU / OTU_2D

Este notebook corresponde a la ejecución completa del pipeline implementado y validado previamente sobre el subconjunto OTU_2D de MMOTU.

El objetivo de esta etapa es preparar y organizar la totalidad de las muestras aceptadas, preservando los archivos originales y generando la información necesaria para su trazabilidad. No se aplican transformaciones geométricas, dado que estas serán definidas posteriormente de acuerdo con las arquitecturas seleccionadas.

> **Prerrequisitos.** Para reproducir esta etapa desde cero es necesario disponer localmente del dataset fuente MMOTU / OTU_2D en `data/raw/MMOTU/OTU_2D/` (`images/`, `annotations/`, `train.txt`, `val.txt`, `train_cls.txt` y `val_cls.txt`). El dataset no se distribuye con el repositorio.
>

## 1. Configuración del entorno

Se cargan los módulos correspondientes al pipeline implementado y se establecen las rutas utilizadas para la ejecución completa sobre el conjunto de datos.

In [1]:
import sys
from pathlib import Path

import pandas as pd

def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto."
    )


PROJECT_ROOT = locate_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.pipeline.config import (PROCESSED_DATA_DIR,)
from src.pipeline.dataset import (load_dataset_index,)
from src.pipeline.validation import (validate_dataset_index,)
from src.pipeline.pipeline import (run_pipeline, save_pipeline_reports,)

## 2. Carga del índice completo

Se construye el índice correspondiente a la totalidad de las muestras disponibles en OTU_2D, manteniendo la información de identificación, etiquetas y particiones proporcionadas por el conjunto original.

In [3]:
dataset_df = load_dataset_index()

print("Número total de muestras:",len(dataset_df))
print("\nDistribución por partición:")
print(dataset_df["split"].value_counts())

dataset_df.head()

Número total de muestras: 1469

Distribución por partición:
split
train         1000
validation     469
Name: count, dtype: int64


,image_id,image_name,split,original_class,binary_label,binary_class,image_path,mask_path
0,658,658.JPG,train,5,0,benign,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...
1,384,384.JPG,train,3,0,benign,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...
2,367,367.JPG,train,3,0,benign,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...
3,730,730.JPG,train,1,0,benign,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...
4,1426,1426.JPG,train,7,1,malignant,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...


## 3. Prevalidación del conjunto completo

Antes de generar el conjunto organizado, se ejecutan las validaciones generales del pipeline sobre el índice completo. Esta comprobación permite detectar archivos faltantes, identificadores duplicados o solapamientos entre las particiones antes de realizar la escritura definitiva de las salidas.

In [4]:
validation_results = validate_dataset_index(dataset_df)
validation_results

{'duplicate_ids': 0,
 'missing_images': 0,
 'missing_masks': 0,
 'train_val_overlap': 0,
 'total_samples': 1469}

In [ ]:
# Comprobación automática de los resultados de validación
assert validation_results["total_samples"] == 1469
assert validation_results["duplicate_ids"] == 0
assert validation_results["missing_images"] == 0
assert validation_results["missing_masks"] == 0
assert validation_results["train_val_overlap"] == 0

print("Prevalidación general completada correctamente.")

Prevalidación general completada correctamente.


## 4. Preparación del directorio de salida

Antes de ejecutar el pipeline completo, se verifica el estado del directorio destinado al conjunto preparado. Esta comprobación permite evitar la mezcla de archivos provenientes de ejecuciones anteriores.

In [6]:
print("Directorio de salida:", PROCESSED_DATA_DIR)

print("¿Existe actualmente?:", PROCESSED_DATA_DIR.exists())

if PROCESSED_DATA_DIR.exists():
    existing_items = list(PROCESSED_DATA_DIR.iterdir())

    print("Elementos existentes:", len(existing_items))

    for item in existing_items[:20]:
        print("-", item.name)

Directorio de salida: C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\data\processed\MMOTU_OTU_2D
¿Existe actualmente?: False


## 5. Ejecución completa del pipeline

Se ejecuta el pipeline sobre la totalidad de las muestras validadas. Durante esta etapa, se realiza la validación individual, caracterización técnica, organización de imágenes y máscaras y generación de los registros necesarios para mantener la trazabilidad.

Los archivos originales son organizados mediante copia directa, sin aplicar redimensionamiento ni recodificación.

In [7]:
execution_metadata_df = run_pipeline(
    dataset_df=dataset_df,
    output_root=PROCESSED_DATA_DIR,
    save_outputs=True,
)

print("Registros generados:", len(execution_metadata_df))

print("\nEstado de las muestras:")
print(execution_metadata_df["status"].value_counts())

Registros generados: 1469

Estado de las muestras:
status
included    1469
Name: count, dtype: int64


## 6. Generación de registros de ejecución

Los resultados de la ejecución son almacenados en archivos tabulares que conservan la información técnica y de trazabilidad de las muestras incluidas y de las muestras excluidas.

In [8]:
report_paths = save_pipeline_reports(metadata_df=execution_metadata_df, output_root=PROCESSED_DATA_DIR,)
report_paths

{'metadata_path': WindowsPath('C:/Users/PC/Desktop/JARITZA - PUCP/26-2/Tesis/proyecto_ia_tumores_ovaricos/data/processed/MMOTU_OTU_2D/metadata.csv'),
 'exclusions_path': WindowsPath('C:/Users/PC/Desktop/JARITZA - PUCP/26-2/Tesis/proyecto_ia_tumores_ovaricos/data/processed/MMOTU_OTU_2D/exclusions.csv')}

In [9]:
# Comprobación de la existencia de los archivos generados
assert (PROCESSED_DATA_DIR / "metadata.csv").exists()
assert (PROCESSED_DATA_DIR / "exclusions.csv").exists()

print("Registros de ejecución guardados correctamente.")

Registros de ejecución guardados correctamente.


## 7. Comprobación básica del conjunto organizado

Se verifica que la cantidad de archivos organizados sea consistente con los registros generados durante la ejecución del pipeline. Los directorios `train` y `validation` verificados aquí corresponden a la organización original de MMOTU (1000 y 469 muestras), no a las particiones experimentales.

In [10]:
train_images_dir = (PROCESSED_DATA_DIR / "train" / "images")
train_masks_dir = (PROCESSED_DATA_DIR / "train" / "masks")
validation_images_dir = (PROCESSED_DATA_DIR / "validation" / "images")
validation_masks_dir = (PROCESSED_DATA_DIR / "validation" / "masks")

train_image_count = len(list(train_images_dir.glob("*")))
train_mask_count = len(list(train_masks_dir.glob("*")))
validation_image_count = len(list(validation_images_dir.glob("*")))
validation_mask_count = len(list(validation_masks_dir.glob("*")))

print("Train - imágenes:", train_image_count)
print("Train - máscaras:", train_mask_count)
print("Validation - imágenes:", validation_image_count)
print("Validation - máscaras:", validation_mask_count)

Train - imágenes: 1000
Train - máscaras: 1000
Validation - imágenes: 469
Validation - máscaras: 469


In [11]:
included_count = (execution_metadata_df["status"] == "included").sum()
excluded_count = (execution_metadata_df["status"] == "excluded").sum()
organized_image_count = (train_image_count + validation_image_count)
organized_mask_count = (train_mask_count + validation_mask_count)

print("Muestras incluidas:", included_count)
print("Muestras excluidas:", excluded_count)
print("Imágenes organizadas:", organized_image_count)
print("Máscaras organizadas:", organized_mask_count)

Muestras incluidas: 1469
Muestras excluidas: 0
Imágenes organizadas: 1469
Máscaras organizadas: 1469
